---
---
# **Tutorial 2B:** *Tools and Function Calling*
### *Giving a local model access to our formulary*
---
---

### QUESTION FOR TODAY
> *A language model has no access to our formulary. So how can it ever tell us a real drug price, or whether a medicine is banned in a particular country?*

### TASK FOR TODAY

**The scenario:** A care coordinator types a plain-English question:

> *"Is Nimesulide banned in India? And what does Metformin cost us?"*

Our job is to build an assistant that answers **from our formulary file** — never from the model's memory.

### OUR ROADMAP
| Step | What we do | The idea |
|---|---|---|
| 1 | **The Problem** | The model guesses — and guesses *dangerously* |
| 2 | **Define the Tools** | A tool is just an ordinary Python function |
| 3 | **Describe the Tools** | The schema: how the model learns what exists |
| 4 | **Function Calling** | The 4-step handshake, one step at a time |
| 5 | **Two Tools** | The model picks the right one by itself |
| 6 | **The Comparison** | Same question, with and without tools |

### THE ONE IDEA BEHIND ALL OF IT
> **ASK → THE MODEL REQUESTS A TOOL → *WE* RUN IT → SEND THE RESULT BACK → THE MODEL ANSWERS.**

---
# **Our Data - *drug_formulary.csv***
---

| Column | Meaning |
|---|---|
| `drug_name` | generic name |
| `price_inr_per_strip`, `pack_size` | what a pack costs (illustrative) |
| `india_status`, `us_status`, `uk_status` | Approved / Restricted / Banned |
| `regulatory_note` | *why* — the reason behind the status |

> ⚠️ **Disclaimer:** *Practice data for learning only. Regulatory lists change constantly — the authoritative source is [CDSCO](https://cdsco.gov.in/opencms/opencms/en/consumer/List-Of-Banned-Drugs/) for India. Never use a teaching file for a real coverage, clinical or compliance decision.*

In [ ]:
# ---------------------------------------------------------
# 1. BRING IN OUR TOOLKITS
# ---------------------------------------------------------
import pandas as pd      # to read our formulary file
import requests          # to talk to the local Ollama server
import json              # to look at the raw responses

# ---------------------------------------------------------
# 2. POINT AT THE LOCAL MODEL (running on YOUR machine)
# ---------------------------------------------------------
OLLAMA_URL = "http://localhost:11434/api/chat"

# Only 3.4 GB, and it supports tool calling.
#   Pull it once with:   ollama pull qwen3.5:4b
MODEL_NAME = "qwen3.5:4b"

# ---------------------------------------------------------
# 3. LOAD THE FORMULARY
# ---------------------------------------------------------
formulary = pd.read_csv("data/drug_formulary.csv")

print("Formulary loaded:", len(formulary), "drugs")
formulary.head()

In [ ]:
# ---------------------------------------------------------
# A QUICK LOOK AT WHAT WE ARE WORKING WITH
# ---------------------------------------------------------
# Notice that a drug's status is NOT the same in every country.
formulary[["drug_name", "india_status", "us_status", "uk_status"]]

Look at the table!

There is **no single right answer** to "is this drug banned?" It depends on the country, and it changes as regulators act. This is exactly the kind of fact a model must never answer from memory.

### WARM UP THE MODEL AND TIME IT

In [ ]:
import time

start = time.time()
requests.post(
    OLLAMA_URL,
    json={
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Say OK"}],
        "stream": False,
        "think": False,        # <- IMPORTANT: turns off slow "thinking" mode
    },
)
print(f"First call (loads the model): {time.time() - start:.1f} seconds")

start = time.time()
requests.post(
    OLLAMA_URL,
    json={
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Say OK"}],
        "stream": False,
        "think": False,
    },
)
print(f"Second call (model warm)    : {time.time() - start:.1f} seconds")

print()
print("If the second call is over ~20 seconds, switch to a smaller model:")
print('   MODEL_NAME = "llama3.2:3b"    (~2 GB, no thinking mode, fastest)')

---
# **Step 1: The Problem - *the model guesses***
---

Let's ask our local model directly, with no tools at all, and watch what happens.

In [ ]:
# ---------------------------------------------------------
# ASKING THE MODEL WITH NO TOOLS AT ALL
# ---------------------------------------------------------
def ask_without_tools(question):
    """Send a plain question to the local model - no tools attached."""
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "messages": [{"role": "user", "content": question}],
            "stream": False,
            "think": False,   # <- turn OFF Qwen thinking mode (much faster)
        },
    ).json()
    return response["message"]["content"]


question = "Is Nimesulide banned in India? Answer in one or two lines."
print(ask_without_tools(question))

Whatever came back, look at it critically!

> ### 🚨 This is the dangerous failure mode for a health company.
> A wrong answer that *sounds* authoritative is worse than no answer at all. And notice — a bigger model does not fix this. The information simply **is not in the model**. It is in our file.

The fix is not a better prompt. The fix is a **tool**.

---
# **Step 2: Define the Tools**
---

Here is the part that surprises everyone:

> ### 🔧 **A "tool" is just an ordinary Python function.**

We need two:

1. `get_drug_price(drug_name)` — what does it cost?
2. `check_drug_status(drug_name, country)` — is it allowed there?

In [ ]:
# ---------------------------------------------------------
# TOOL 1: PRICE LOOKUP
# ---------------------------------------------------------
def get_drug_price(drug_name: str):
    """Look up the price of a drug in our formulary."""

    # Find the row. .lower() and .strip() make us forgiving about
    # capitals and stray spaces in whatever the model sends us.
    row = formulary[formulary["drug_name"].str.lower() == drug_name.lower().strip()]

    # ALWAYS handle "not found" - the model will read this text too
    if row.empty:
        return f"'{drug_name}' is not in our formulary."

    drug = row.iloc[0]

    # A price of 0 in our file means the drug is not marketed
    if drug["price_inr_per_strip"] == 0:
        return (f"{drug['drug_name']} is not marketed "
                f"(status in India: {drug['india_status']}). No price available.")

    return f"{drug['drug_name']}: Rs {drug['price_inr_per_strip']} per {drug['pack_size']}"

In [ ]:
# ---------------------------------------------------------
# TOOL 2: REGULATORY STATUS LOOKUP
# ---------------------------------------------------------
# The model might say "USA", "US" or "United States" - we accept all of them.
COUNTRY_COLUMNS = {
    "india":         ("india_status", "India"),
    "us":            ("us_status",    "USA"),
    "usa":           ("us_status",    "USA"),
    "united states": ("us_status",    "USA"),
    "uk":            ("uk_status",    "UK"),
    "united kingdom":("uk_status",    "UK"),
}


def check_drug_status(drug_name: str, country: str):
    """Check whether a drug is approved, restricted or banned in a country."""

    row = formulary[formulary["drug_name"].str.lower() == drug_name.lower().strip()]
    if row.empty:
        return f"'{drug_name}' is not in our formulary."

    # Look up which column to read for this country
    lookup = COUNTRY_COLUMNS.get(country.lower().strip())
    if lookup is None:
        return f"We only hold regulatory data for India, USA and UK - not '{country}'."

    status_column, country_label = lookup
    drug = row.iloc[0]

    return (f"{drug['drug_name']} in {country_label}: {drug[status_column]}. "
            f"Note: {drug['regulatory_note']}")

In [ ]:
# ---------------------------------------------------------
# TEST THEM LIKE NORMAL FUNCTIONS - no AI involved yet!
# ---------------------------------------------------------
print(get_drug_price("Metformin"))
print(get_drug_price("Rosiglitazone"))        # banned -> no price
print(get_drug_price("Aspirin"))              # not in our file
print()
print(check_drug_status("Nimesulide", "India"))
print()
print(check_drug_status("Nimesulide", "USA"))

Notice two things:

1. **The AI has not been involved at all yet.**
2. **Every failure returns a clear sentence**, never a crash.

---
# **Step 3: Describe the Tools to the Model**
---

The model cannot read our Python file. It only reads text. So we hand it a **schema** describing each tool: its `name`, its `description`, and the `parameters` it needs.

> ### ⚠️ The `description` is the most important line you will write.
> It is the *only* thing the model uses to decide which tool to reach for. Vague description → wrong tool, or no tool at all.

In [ ]:
# ---------------------------------------------------------
# THE TOOL MENU (the format Ollama expects)
# ---------------------------------------------------------
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_drug_price",                       # must match our function
            "description": (                                # WHEN to use it
                "Get the price of a drug from the company formulary. "
                "Use this whenever the user asks about the cost or price of a medicine."
            ),
            "parameters": {                                 # WHAT it needs
                "type": "object",
                "properties": {
                    "drug_name": {
                        "type": "string",
                        "description": "The generic name of the drug, e.g. Metformin",
                    },
                },
                "required": ["drug_name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_drug_status",
            "description": (
                "Check whether a drug is approved, restricted or banned in a given "
                "country. Use this for any question about legality, bans, or "
                "regulatory status of a medicine."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "drug_name": {
                        "type": "string",
                        "description": "The generic name of the drug",
                    },
                    "country": {
                        "type": "string",
                        "description": "The country to check: India, USA or UK",
                    },
                },
                "required": ["drug_name", "country"],
            },
        },
    },
]

# ---------------------------------------------------------
# THE TOOL REGISTRY: tool name -> the real Python function
# ---------------------------------------------------------
# Cleaner than a pile of if/elif, and it scales to 50 tools
# without ever touching the calling code.
TOOL_REGISTRY = {
    "get_drug_price":    get_drug_price,
    "check_drug_status": check_drug_status,
}

print("Tools on the menu:", list(TOOL_REGISTRY.keys()))

---
# **Step 4: Function Calling - *the 4-step handshake***
---

> ### 🚨 **The model does NOT run our code.**
> It cannot reach our files or our database. All it can do is send back a **request** saying *"please run `check_drug_status` with these arguments."*
> **Our code decides whether to obey.** That gap is our security boundary — and for a health company handling member data, it is the whole reason this pattern is safe to deploy.

One question becomes four steps:

| | Step | Who does it |
|---|---|---|
| 1️⃣ | Send the question **+ the tool menu** | us → model |
| 2️⃣ | Model replies: *"call this tool with these arguments"* | model → us |
| 3️⃣ | **Run the function** | **our code** |
| 4️⃣ | Send the result back; model writes the answer | us → model → us |

In [ ]:
# ---------------------------------------------------------
# STEP 1 of 4: SEND THE QUESTION + THE TOOL MENU
# ---------------------------------------------------------
# The system prompt sets the ground rule: look it up, never guess.
SYSTEM_PROMPT = (
    "You are a formulary assistant for a health services company. "
    "Always use the provided tools to look up drug prices and regulatory status. "
    "Never guess or rely on your own knowledge about drugs."
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": "Is Nimesulide banned in India?"},
]

response = requests.post(
    OLLAMA_URL,
    json={
        "model": MODEL_NAME,
        "messages": messages,
        "tools": TOOLS,          # <-- here is the menu card
        "stream": False,
        "think": False,   # <- turn OFF Qwen thinking mode (much faster)
    },
).json()

message = response["message"]

# Notice: 'content' is EMPTY. The model did not answer -
# it asked us to do something first.
print("Model's text answer:", repr(message.get("content", "")))
print()
print("What it sent instead:")
print(json.dumps(message.get("tool_calls", []), indent=2))

In [ ]:
# ---------------------------------------------------------
# STEP 2 of 4: READ THE MODEL'S REQUEST
# ---------------------------------------------------------
tool_call = message["tool_calls"][0]["function"]

tool_name = tool_call["name"]          # which function it wants
tool_args = tool_call["arguments"]     # the arguments it extracted

print("The model is REQUESTING a tool call:")
print("   Tool name :", tool_name)
print("   Arguments :", tool_args)

We typed *"Is Nimesulide banned in India?"* — ordinary human language. The model read it and produced clean, structured arguments: `drug_name='Nimesulide'`, `country='India'`.

In [ ]:
# ---------------------------------------------------------
# STEP 3 of 4: **WE** RUN THE FUNCTION
# ---------------------------------------------------------
# Look up the function by name in our registry, then call it.
# The ** unpacks {"drug_name": "Nimesulide", ...} into named arguments.
function_to_run = TOOL_REGISTRY[tool_name]
result = function_to_run(**tool_args)

print(f"We ran {tool_name}() ourselves, on our own machine.")
print("Tool result:", result)

In [ ]:
# ---------------------------------------------------------
# STEP 4 of 4: SEND THE RESULT BACK, GET THE FINAL ANSWER
# ---------------------------------------------------------
# Add the model's request, then our tool's answer, to the conversation
messages.append(message)                                  # what the model asked for
messages.append({"role": "tool", "content": str(result)})  # what our tool returned

final_response = requests.post(
    OLLAMA_URL,
    json={
        "model": MODEL_NAME,
        "messages": messages,
        "tools": TOOLS,
        "stream": False,
        "think": False,   # <- turn OFF Qwen thinking mode (much faster)
    },
).json()

print("Final answer from the model:\n")
print(final_response["message"]["content"])

**That is function calling.** The whole idea, complete.

The model supplied the *language understanding* at both ends — reading the messy question, then writing a human sentence around our raw data — and **our code supplied the truth in the middle.** That regulatory status came from our CSV, not from the model's memory.

---
# **Step 5: Two Tools - *the model routes by itself***
---

Let's wrap those four steps into one reusable function — this is the shape of real application code — and then watch the model choose between our two tools on its own.

In [ ]:
# ---------------------------------------------------------
# THE WHOLE HANDSHAKE, PACKAGED INTO ONE REUSABLE FUNCTION
# ---------------------------------------------------------
def ask(user_question):
    """Send a question, handle any tool call, and return the final answer."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_question},
    ]

    # --- Step 1: ask, with the tool menu ---
    response = requests.post(
        OLLAMA_URL,
        json={"model": MODEL_NAME, "messages": messages,
              "tools": TOOLS, "stream": False, "think": False},
    ).json()

    message = response["message"]

    # --- If the model did not want a tool, it already answered ---
    if "tool_calls" not in message:
        print("(No tool needed)")
        return message.get("content", "")

    # --- Step 2: read the request ---
    tool_call = message["tool_calls"][0]["function"]
    tool_name = tool_call["name"]
    tool_args = tool_call["arguments"]
    print(f"🔧 Model requested : {tool_name}({tool_args})")

    # --- Step 3: we run it ---
    result = TOOL_REGISTRY[tool_name](**tool_args)
    print(f"📦 Tool returned   : {result}")

    # --- Step 4: send the result back ---
    messages.append(message)
    messages.append({"role": "tool", "content": str(result)})

    final = requests.post(
        OLLAMA_URL,
        json={"model": MODEL_NAME, "messages": messages,
              "tools": TOOLS, "stream": False, "think": False},
    ).json()

    return final["message"]["content"]

In [ ]:
# ---------------------------------------------------------
# TEST 1: a PRICE question
# ---------------------------------------------------------
print(ask("What does Metformin cost us?"))

In [ ]:
# ---------------------------------------------------------
# TEST 2: a REGULATORY question (same code, different tool chosen!)
# ---------------------------------------------------------
print(ask("Is Metamizole allowed in the UK?"))

In [ ]:
# ---------------------------------------------------------
# TEST 3: a drug that is NOT in our formulary
# ---------------------------------------------------------
# Our function returns a clear sentence instead of crashing -
# and the model reads that sentence and reports it honestly.
print(ask("What is the price of Aspirin?"))

We changed **nothing but the question**, and the model routed to a different tool each time — purely from the `description` we wrote.

And on the last one, our tool said *"not in our formulary"*, the model read that, and told the truth instead of inventing a price. **Your error messages are part of your interface.**

---
# **Step 6: The Comparison**
---

Same question. Same model. Same laptop. The only difference is whether it had a tool.

In [ ]:
# ---------------------------------------------------------
# THE SIDE-BY-SIDE THAT MAKES THE CASE
# ---------------------------------------------------------
question = "Is Nimesulide banned in India? Answer in one or two lines."

print("=" * 65)
print("WITHOUT TOOLS (the model guesses from memory):")
print("=" * 65)
print(ask_without_tools(question))

print()
print("=" * 65)
print("WITH TOOLS (grounded in our formulary):")
print("=" * 65)
print(ask(question))

---
# **What We Built Today**
---

We started with a model that confidently guessed about drug regulation — the most dangerous thing it could do in a health setting. We finished with one that reads a plain-English question, picks the right function, looks up **our** formulary, and answers from real data.

The loop behind all of it:

> **ASK → THE MODEL REQUESTS A TOOL → *WE* RUN IT → SEND THE RESULT BACK → THE MODEL ANSWERS.**

Three things worth carrying back to work:

1. **A tool is just a function.** Ordinary code you can test, log and version like anything else.
2. **The `description` is the interface.** The model chooses by reading your words.
3. **The model requests; our code executes.** That gap is where validation, permissions and audit logging belong — and with member data, that gap is not optional.

And notice what we did *not* need today: no API key, no rate limit, no data leaving the machine. For regulated data, **local models plus local tools** is a genuinely serious architecture, not a toy.

Our tools today only *read*. The moment a tool can *write* — update a record, approve a refill, send a message to a member — the same four steps become a system that acts. That is where a human approval step stops being optional.

---
# **Thank you !**
---

Author: *Aneetta Sara Shany*

Date: *2026 August 05, Wednesday*